# Fraud Detection App - Notebook Version
This notebook mirrors the logic of the Streamlit app for loading the model, encoder, and testing predictions on a dataset.

In [ ]:
import pandas as pd
import joblib
from geopy.distance import geodesic
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Load model and encoders
model = joblib.load('fraud_detection_model.jb')
encoder = joblib.load('label_encoder.jb')

In [ ]:
# Load dataset
df = pd.read_csv('transactions.csv')
df.head()

In [ ]:
# Compute distance using geodesic
df['distance'] = df.apply(lambda row: geodesic((row['lat'], row['long']), (row['merch_lat'], row['merch_long'])).km, axis=1)

In [ ]:
# Process and encode features
df['cc_num'] = df['cc_num'].apply(lambda x: hash(str(x)) % (10 ** 2))
df['gender'] = df['gender'].str.capitalize()
categorical_cols = ['merchant', 'category', 'gender']
for col in categorical_cols:
    df[col] = df[col].apply(lambda x: x if x in encoder[col].classes_ else '<unknown>')
    if '<unknown>' not in encoder[col].classes_:
        encoder[col].classes_ = list(encoder[col].classes_) + ['<unknown>']
    df[col] = encoder[col].transform(df[col])

In [ ]:
# Prepare input features
X = df[['merchant', 'category', 'amt', 'distance', 'hour', 'day', 'month', 'gender', 'cc_num']]

In [ ]:
# Predict fraudulence
df['predicted'] = model.predict(X)
df[['merchant', 'category', 'amt', 'predicted']].head(10)